# GitHub Tool for Snowflake Intelligence

This notebook sets up the `github_tool_sproc` stored procedure, which allows your Snowflake Intelligence / Cortex agents to query GitHub repositories in read-only mode.

**Supported Actions:**
- `list_issues` — List open issues in a repository
- `get_issue` — Get details of a specific issue
- `list_prs` — List open pull requests in a repository
- `get_file` — Fetch the contents of a file from a repository
- `search_issues` — Search issues/PRs by keyword

Run each cell **one at a time** in order.

In [ ]:
# Get the active Snowflake session
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
-- Create a dedicated database and schema for the tool
-- Replace YOUR_DB and YOUR_SCHEMA with your preferred names
USE ROLE ACCOUNTADMIN;

CREATE DATABASE IF NOT EXISTS YOUR_DB;
CREATE SCHEMA IF NOT EXISTS YOUR_DB.YOUR_SCHEMA;

USE DATABASE YOUR_DB;
USE SCHEMA YOUR_SCHEMA;

In [ ]:
-- Create a network rule to allow outbound access to the GitHub API
USE ROLE ACCOUNTADMIN;
USE DATABASE YOUR_DB;
USE SCHEMA YOUR_SCHEMA;

CREATE OR REPLACE NETWORK RULE github_api_rule
    MODE = EGRESS
    TYPE = HOST_PORT
    VALUE_LIST = ('api.github.com:443');

In [ ]:
-- Store your GitHub Personal Access Token securely as a Snowflake secret
-- Generate a Classic token at: https://github.com/settings/tokens
-- Required scope: 'public_repo' for public repos, full 'repo' for private repos
USE ROLE ACCOUNTADMIN;
USE DATABASE YOUR_DB;
USE SCHEMA YOUR_SCHEMA;

CREATE OR REPLACE SECRET github_api_token
    TYPE = GENERIC_STRING
    SECRET_STRING = '<your-github-pat-token>';

In [ ]:
-- Create an external access integration linking the network rule and secret
USE ROLE ACCOUNTADMIN;
USE DATABASE YOUR_DB;
USE SCHEMA YOUR_SCHEMA;

CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION github_access_integration
    ALLOWED_NETWORK_RULES = (github_api_rule)
    ALLOWED_AUTHENTICATION_SECRETS = (github_api_token)
    ENABLED = TRUE;

In [ ]:
-- Grant usage on the integration to your role
-- Replace PUBLIC with a more specific role if needed
USE ROLE ACCOUNTADMIN;

GRANT USAGE ON INTEGRATION github_access_integration TO ROLE PUBLIC;

In [ ]:
-- Create a stage to store the stored procedure code
USE ROLE ACCOUNTADMIN;
USE DATABASE YOUR_DB;
USE SCHEMA YOUR_SCHEMA;

CREATE STAGE IF NOT EXISTS custom_tools;
GRANT USAGE ON STAGE custom_tools TO ROLE PUBLIC; -- Replace PUBLIC with your role if needed

In [ ]:
from snowflake.snowpark.functions import sproc
from snowflake.snowpark.types import StringType
from snowflake.snowpark import Session

session.add_packages('requests')
database = session.get_current_database()
schema = session.get_current_schema()

@sproc(
    name="github_tool_sproc",
    replace=True,
    is_permanent=True,
    stage_location="@custom_tools",
    packages=['snowflake-snowpark-python', 'requests'],
    external_access_integrations=['github_access_integration'],
    secrets={'api_token': 'github_api_token'}
)
def github_tool(session: Session, action: str, repo: str, query: str = "") -> str:
    import requests
    import base64
    import _snowflake

    # Retrieve the GitHub PAT from Snowflake secret
    token = _snowflake.get_generic_secret_string('api_token')

    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28"
    }

    base_url = "https://api.github.com"

    def format_issues(issues):
        if not issues:
            return "No issues found."
        lines = []
        for issue in issues[:10]:
            lines.append(
                f"#{issue['number']} [{issue['state'].upper()}] {issue['title']}\\n"
                f"  Author: {issue['user']['login']} | "
                f"Comments: {issue['comments']} | "
                f"URL: {issue['html_url']}"
            )
        return "\\n\\n".join(lines)

    def format_prs(prs):
        if not prs:
            return "No pull requests found."
        lines = []
        for pr in prs[:10]:
            lines.append(
                f"#{pr['number']} [{pr['state'].upper()}] {pr['title']}\\n"
                f"  Author: {pr['user']['login']} | "
                f"Branch: {pr['head']['ref']} → {pr['base']['ref']} | "
                f"URL: {pr['html_url']}"
            )
        return "\\n\\n".join(lines)

    try:
        # ── list_issues ──────────────────────────────────────────────
        if action == "list_issues":
            url = f"{base_url}/repos/{repo}/issues"
            params = {"state": "open", "per_page": 10}
            if query:
                params["labels"] = query
            resp = requests.get(url, headers=headers, params=params)
            resp.raise_for_status()
            issues = [i for i in resp.json() if "pull_request" not in i]
            return f"Open Issues in {repo}:\\n\\n" + format_issues(issues)

        # ── get_issue ────────────────────────────────────────────────
        elif action == "get_issue":
            if not query.isdigit():
                return "For get_issue, 'query' must be the issue number (e.g. '42')."
            url = f"{base_url}/repos/{repo}/issues/{query}"
            resp = requests.get(url, headers=headers)
            resp.raise_for_status()
            issue = resp.json()
            return (
                f"Issue #{issue['number']}: {issue['title']}\\n"
                f"State: {issue['state'].upper()}\\n"
                f"Author: {issue['user']['login']}\\n"
                f"Created: {issue['created_at'][:10]}\\n"
                f"Labels: {', '.join(l['name'] for l in issue['labels']) or 'None'}\\n"
                f"Comments: {issue['comments']}\\n"
                f"URL: {issue['html_url']}\\n\\n"
                f"Description:\\n{issue['body'] or 'No description provided.'}"
            )

        # ── list_prs ─────────────────────────────────────────────────
        elif action == "list_prs":
            url = f"{base_url}/repos/{repo}/pulls"
            params = {"state": "open", "per_page": 10}
            resp = requests.get(url, headers=headers, params=params)
            resp.raise_for_status()
            return f"Open Pull Requests in {repo}:\\n\\n" + format_prs(resp.json())

        # ── get_file ─────────────────────────────────────────────────
        elif action == "get_file":
            if not query:
                return "For get_file, 'query' must be the file path (e.g. 'src/main.py')."
            url = f"{base_url}/repos/{repo}/contents/{query}"
            resp = requests.get(url, headers=headers)
            resp.raise_for_status()
            data = resp.json()
            if data.get("encoding") == "base64":
                content = base64.b64decode(data["content"]).decode("utf-8", errors="replace")
                if len(content) > 3000:
                    content = content[:3000] + "\\n\\n... [truncated — file too large]"
                return f"File: {data['path']} ({data['size']} bytes)\\n\\n{content}"
            return f"Could not decode file: {data.get('type', 'unknown type')}"

        # ── search_issues ────────────────────────────────────────────
        elif action == "search_issues":
            if not query:
                return "For search_issues, 'query' must contain a search keyword."
            url = f"{base_url}/search/issues"
            params = {"q": f"{query} repo:{repo}", "per_page": 10}
            resp = requests.get(url, headers=headers, params=params)
            resp.raise_for_status()
            items = resp.json().get("items", [])
            if not items:
                return f"No issues or PRs found in {repo} matching '{query}'."
            return f"Search results for '{query}' in {repo}:\\n\\n" + format_issues(items)

        else:
            return (
                f"Unknown action: '{action}'.\\n"
                "Supported actions: list_issues, get_issue, list_prs, get_file, search_issues"
            )

    except requests.exceptions.HTTPError as e:
        status = e.response.status_code if e.response else "unknown"
        return f"GitHub API error (HTTP {status}): {str(e)}"
    except Exception as e:
        return f"Unexpected error: {str(e)}"

session.sql("SHOW PROCEDURES LIKE 'github_tool_sproc'").collect()

In [ ]:
-- Test 1: List open issues
CALL github_tool_sproc('list_issues', 'Snowflake-Labs/snowflake-intelligence-awesome-tools', '');

In [ ]:
-- Test 2: Get a specific issue by number
CALL github_tool_sproc('get_issue', 'Snowflake-Labs/snowflake-intelligence-awesome-tools', '1');

In [ ]:
-- Test 3: List open pull requests
CALL github_tool_sproc('list_prs', 'Snowflake-Labs/snowflake-intelligence-awesome-tools', '');

In [ ]:
-- Test 4: Fetch a file from a repo
CALL github_tool_sproc('get_file', 'Snowflake-Labs/snowflake-intelligence-awesome-tools', 'README.md');

In [ ]:
-- Test 5: Search issues by keyword
CALL github_tool_sproc('search_issues', 'Snowflake-Labs/snowflake-intelligence-awesome-tools', 'tool');